In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path(
    "/srv/scratch/z5297792/aviso_eddy_dataset/processed/eddy_dataset_processed.parquet"
)
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Processed dataset not found: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Eddy", "Day"]).reset_index(drop=True)
print(f"Loaded {len(df):,} eddy-day rows from {DATA_PATH}")
df.head()

Loaded 453,309 eddy-day rows from /srv/scratch/z5297792/aviso_eddy_dataset/processed/eddy_dataset_processed.parquet


,Eddy,Day,Date,Cyc,lon,lat,ic,jc,xc,yc,...,Omega,q11,q12,q22,Rc,psi0,AR,R,Age,source_file
0,1,16072,1994-01-02,AE,149.388784,-41.813765,35,9,397.941074,124.679076,...,0.000004,1.776830,-0.465120,0.687729,125.323701,-28.027626,1.942958,75.590039,47,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...
1,1,16073,1994-01-03,AE,149.403113,-41.758355,35,9,399.259096,130.826827,...,0.000004,1.930643,-0.550539,0.684909,125.015516,-27.439127,2.118809,75.675331,47,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...
2,1,16074,1994-01-04,AE,149.466532,-41.617063,35,11,405.092506,146.503209,...,0.000003,1.923138,-0.557851,0.691597,124.707331,-26.850629,2.118393,75.339928,47,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...
3,1,16075,1994-01-05,AE,149.529951,-41.475772,36,12,410.925917,162.179590,...,0.000003,1.915632,-0.565163,0.698284,124.399146,-26.262130,2.118317,75.004526,47,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...
4,1,16076,1994-01-06,AE,149.576872,-41.354687,36,13,415.241842,175.614070,...,0.000003,1.808704,-0.508663,0.712941,124.090962,-25.673632,1.978232,74.989772,47,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...


In [ ]:
def day_plot(day, df_data, num_label=True):

    fnumber = 1461 + ((day - 1462) // 30)*30
    fname = f'/srv/scratch/z3533156/26year_BRAN2020/outer_avg_{fnumber:05}.nc'
    with nc.Dataset(fname) as ds:
        ocean_time = ds['ocean_time'][:] / 86400
        t = np.where(ocean_time == day)[0][0]
        ut = ds['u_eastward'][t, -1, :, :].T
        vt = ds['v_northward'][t, -1, :, :].T

    df_day = df_data.loc[df_data.Day.eq(day)].copy()

    cs = np.hypot(ut, vt)

    fig, ax = plt.subplots(figsize=(8, 10))
    im = ax.pcolor(X_grid, Y_grid, cs, shading='nearest', vmin=0, vmax=2.5, cmap='Blues_r')
    fig.colorbar(im, ax=ax, label=r'Current speed (ms$^{-1}$)')

    clrs = np.where(df_day.Cyc.eq('CE'), 'c', 'r')
    ax.scatter(df_day.xc, df_day.yc, c=clrs, edgecolors='k', linewidths=0.8, s=60, zorder=10)

    if 'Q' not in df_day.columns:
        df_day['Q'] = list(
            np.stack([
                np.stack([df_day.q11.values, df_day.q12.values], axis=1),
                np.stack([df_day.q12.values, df_day.q22.values], axis=1)
            ], axis=1)
        )

    for xc, yc, e, Q, Rc, cyc in zip(df_day.xc, df_day.yc, df_day.Eddy, df_day.Q, df_day.Rc, df_day.Cyc):

        # ----- Where I plot the eddy's maximum tangenital velocity contour -----
        dx_ell, dy_ell = X_grid - xc, Y_grid - yc
        rho2_ell = Q[0,0]*dx_ell**2 + 2*Q[1,0]*dx_ell*dy_ell + Q[1,1]*dy_ell**2 # rho^2
        ax.contour(X_grid, Y_grid, rho2_ell, levels=[Rc**2/2], colors='r' if cyc=='AE' else 'c')

        if num_label:
            ax.annotate(
                str(e), (xc, yc),
                textcoords='offset points', xytext=(3, 3),
                fontsize=12, color='w', weight='bold',
                path_effects=[pe.withStroke(linewidth=2, foreground='k')],
                zorder=11
            )

    c1 = ax.contour(X_grid, Y_grid, lat_rho, levels=[-40, -35, -30, -25], colors='k', linewidths=.5)
    ax.clabel(c1, fmt=lambda v: f"{np.abs(v):.0f}°S", inline=True, colors='k')
    c2 = ax.contour(X_grid, Y_grid, lon_rho, levels=[150, 155, 160], colors='k', linewidths=.5)
    ax.clabel(c2, fmt=lambda v: f"{v:.0f}°E", inline=True, colors='k')
                
    ax.set_title(f'Day {day} | {pd.Timestamp("1990-01-01") + pd.Timedelta(days=day)}')
    ax.set_aspect('equal', adjustable='datalim')
    ax.set_xlabel('x (km)')
    ax.set_ylabel('y (km)')
    ax.set_xlim(x_grid.min(), x_grid.max())
    ax.set_ylim(y_grid.min(), y_grid.max())


In [ ]:
day_plot(day, df_eddies, num_label=True)